# AEMO NEM Dispatch Data Fetch and Analysis

import io
import zipfile
from datetime import date, timedelta
import pandas as pd
import httpx
import asyncio
import json

In [5]:
import io
import zipfile
from datetime import date, timedelta
import pandas as pd
import httpx

## 1. Fetching Data from AEMO NEM Archive

This section fetches the daily Dispatch Interval data from the AEMO (Australian Energy Market Operator) NEM (National Electricity Market) archive. The data is typically available with a two-day delay.

In [6]:
day = date.today() - timedelta(days=2)  # give the Archive time to publish
url = (
    "https://www.nemweb.com.au/Reports/ARCHIVE/DispatchIS_Reports/"
    f"PUBLIC_DISPATCHIS_{day.strftime('%Y%m%d')}.zip"
)


async def fetch_and_extract_csv(url):
    async with httpx.AsyncClient(
        timeout=60, headers={"User-Agent": "Mozilla/5.0"}
    ) as client:
        resp = await client.get(url)
    resp.raise_for_status()
    print("url:", url)
    print("status:", resp.status_code, "| bytes:", len(resp.content))

    outer = zipfile.ZipFile(io.BytesIO(resp.content))
    names = outer.namelist()
    print(f"outer zip: {len(names)} nested per-interval zips, e.g. {names[:3]}")

    if not names:
        raise ValueError("No nested zip files found in the archive.")

    inner_bytes = outer.read(names[0])
    inner = zipfile.ZipFile(io.BytesIO(inner_bytes))
    csv_name = next((n for n in inner.namelist() if n.upper().endswith(".CSV")), None)

    if csv_name is None:
        raise ValueError("No CSV file found in the first nested zip.")

    csv_text = inner.read(csv_name).decode("utf-8", errors="replace")
    print(f"\none nested zip -> {csv_name!r}, first 6 lines of the real MMS CSV:")
    for line in csv_text.splitlines()[:6]:
        print(" ", line)
    return csv_text

## 2. Parsing and Loading Data into DataFrame

The raw CSV from AEMO contains multiple tables. This code specifically extracts the `DISPATCH,CASE_SOLUTION` table and loads it into a pandas DataFrame.

In [7]:
try:
    csv_raw_text = await fetch_and_extract_csv(url)

    # Process the multi-table CSV content
    lines = csv_raw_text.splitlines()

    case_solution_header = []
    case_solution_data = []

    for line in lines:
        parts = line.split(",")
        if len(parts) > 3:
            record_type = parts[0]
            table_name = parts[2]  # e.g., 'CASE_SOLUTION'

            if (
                record_type == "I"
                and parts[1] == "DISPATCH"
                and table_name == "CASE_SOLUTION"
            ):
                # Extract column names, skipping the first 4 metadata fields
                case_solution_header = parts[4:]
            elif (
                record_type == "D"
                and parts[1] == "DISPATCH"
                and table_name == "CASE_SOLUTION"
            ):
                # Collect data rows, skipping the first 4 metadata fields
                case_solution_data.append(",".join(parts[4:]))

    if not case_solution_header or not case_solution_data:
        raise ValueError(
            "Could not find 'DISPATCH,CASE_SOLUTION' header or data in the CSV."
        )

    # Combine header and data for parsing
    parsed_csv_content = (
        ",".join(case_solution_header) + "\n" + "\n".join(case_solution_data)
    )

    df = pd.read_csv(io.StringIO(parsed_csv_content))
    print(
        "\nData loaded into pandas DataFrame for 'DISPATCH,CASE_SOLUTION'. First 5 rows:"
    )
    display(df.head())
except Exception as exc:
    print(f"aemo-nem fetch failed ({type(exc).__name__}): {exc}")

url: https://www.nemweb.com.au/Reports/ARCHIVE/DispatchIS_Reports/PUBLIC_DISPATCHIS_20260803.zip
status: 200 | bytes: 5686207
outer zip: 288 nested per-interval zips, e.g. ['PUBLIC_DISPATCHIS_202608030005_0000000530591757.zip', 'PUBLIC_DISPATCHIS_202608030010_0000000530592271.zip', 'PUBLIC_DISPATCHIS_202608030015_0000000530592833.zip']

one nested zip -> 'PUBLIC_DISPATCHIS_202608030005_0000000530591757.CSV', first 6 lines of the real MMS CSV:
  C,NEMP.WORLD,DISPATCHIS,AEMO,PUBLIC,2026/08/03,00:00:08,0000000530591757,DISPATCHIS,0000000530591756
  I,DISPATCH,CASE_SOLUTION,2,SETTLEMENTDATE,RUNNO,INTERVENTION,CASESUBTYPE,SOLUTIONSTATUS,SPDVERSION,NONPHYSICALLOSSES,TOTALOBJECTIVE,TOTALAREAGENVIOLATION,TOTALINTERCONNECTORVIOLATION,TOTALGENERICVIOLATION,TOTALRAMPRATEVIOLATION,TOTALUNITMWCAPACITYVIOLATION,TOTAL5MINVIOLATION,TOTALREGVIOLATION,TOTAL6SECVIOLATION,TOTAL60SECVIOLATION,TOTALASPROFILEVIOLATION,TOTALFASTSTARTVIOLATION,TOTALENERGYOFFERVIOLATION,LASTCHANGED,SWITCHRUNINITIALSTATUS,SWITCH

,SETTLEMENTDATE,RUNNO,INTERVENTION,CASESUBTYPE,SOLUTIONSTATUS,SPDVERSION,NONPHYSICALLOSSES,TOTALOBJECTIVE,TOTALAREAGENVIOLATION,TOTALINTERCONNECTORVIOLATION,...,TOTALREGVIOLATION,TOTAL6SECVIOLATION,TOTAL60SECVIOLATION,TOTALASPROFILEVIOLATION,TOTALFASTSTARTVIOLATION,TOTALENERGYOFFERVIOLATION,LASTCHANGED,SWITCHRUNINITIALSTATUS,SWITCHRUNBESTSTATUS,SWITCHRUNBESTSTATUS_INT
0,2026/08/03 00:05:00,1,0,NaN,1,NaN,0,-7.190496e+07,0,0,...,NaN,NaN,NaN,0,0,0,2026/08/03 00:00:02,1,1,NaN


## 3. Displaying DataFrame as Pretty JSON

This section converts the DataFrame into a JSON format with indentation for better readability.

In [8]:
# Convert DataFrame to JSON string
json_str = df.to_json(orient="records", indent=2)

# Print the pretty JSON string
print(json_str)

[
  {
    "SETTLEMENTDATE":"2026\/08\/03 00:05:00",
    "RUNNO":1,
    "INTERVENTION":0,
    "CASESUBTYPE":null,
    "SOLUTIONSTATUS":1,
    "SPDVERSION":null,
    "NONPHYSICALLOSSES":0,
    "TOTALOBJECTIVE":-71904964.4370000064,
    "TOTALAREAGENVIOLATION":0,
    "TOTALINTERCONNECTORVIOLATION":0,
    "TOTALGENERICVIOLATION":0,
    "TOTALRAMPRATEVIOLATION":0,
    "TOTALUNITMWCAPACITYVIOLATION":0.2,
    "TOTAL5MINVIOLATION":null,
    "TOTALREGVIOLATION":null,
    "TOTAL6SECVIOLATION":null,
    "TOTAL60SECVIOLATION":null,
    "TOTALASPROFILEVIOLATION":0,
    "TOTALFASTSTARTVIOLATION":0,
    "TOTALENERGYOFFERVIOLATION":0,
    "LASTCHANGED":"2026\/08\/03 00:00:02",
    "SWITCHRUNINITIALSTATUS":1,
    "SWITCHRUNBESTSTATUS":1,
    "SWITCHRUNBESTSTATUS_INT":null
  }
]


## 4. Explanation of DataFrame Columns

Based on the AEMO NEM Dispatch data format, here's what each column in the `DISPATCH,CASE_SOLUTION` table represents:

*   **SETTLEMENTDATE**: The date and time for which the dispatch solution applies (e.g., a 5-minute dispatch interval).
*   **RUNNO**: The run number for the dispatch solution. Multiple runs might occur for a single interval.
*   **INTERVENTION**: Indicates if AEMO intervened in the market (e.g., 0 for no intervention, 1 for intervention).
*   **CASESUBTYPE**: Further classification of the case if it's an intervention or specific market event.
*   **SOLUTIONSTATUS**: The status of the dispatch solution (e.g., 1 for successful).
*   **SPDVERSION**: The version of the Short Term Projected Assessment of System Adequacy (SPD) software used.
*   **NONPHYSICALLOSSES**: Losses in the system not related to physical constraints.
*   **TOTALOBJECTIVE**: The total objective function value of the dispatch run. This is a key metric for the market optimization.
*   **TOTALAREAGENVIOLATION**: Total violation of area generation constraints.
*   **TOTALINTERCONNECTORVIOLATION**: Total violation of interconnector constraints.
*   **TOTALGENERICVIOLATION**: Total violation of generic constraints.
*   **TOTALRAMPRATEVIOLATION**: Total violation of ramp rate limits for generators.
*   **TOTALUNITMWCAPACITYVIOLATION**: Total violation of individual unit's MW capacity limits.
*   **TOTAL5MINVIOLATION**: Total violation related to 5-minute dispatch.
*   **TOTALREGVIOLATION**: Total violation related to regulation services.
*   **TOTAL6SECVIOLATION**: Total violation related to 6-second frequency control ancillary services (FCAS).
*   **TOTAL60SECVIOLATION**: Total violation related to 60-second FCAS.
*   **TOTALASPROFILEVIOLATION**: Total violation related to ancillary services profiles.
*   **TOTALFASTSTARTVIOLATION**: Total violation related to fast start units.
*   **TOTALENERGYOFFERVIOLATION**: Total violation related to energy offers.
*   **LASTCHANGED**: The timestamp when this record was last changed.
*   **SWITCHRUNINITIALSTATUS**: Initial status of a switch run (related to market system operations).
*   **SWITCHRUNBESTSTATUS**: Best status achieved by a switch run.
*   **SWITCHRUNBESTSTATUS_INT**: Integer representation of the best switch run status.